# Course 08 lab — measurable NFRs for agentic systems

Northstar's AI-2219 broker-response pilot is moving toward a Commercial Underwriting rollout. We will turn vague production adjectives into a versioned contract, measure a synthetic workload, inject failures, and preserve a blocked production decision where evidence or authority is missing.

**Safety boundary:** everything here is deterministic and credential-free. Workloads, owner decisions, runtime events, costs, and quality predictions are fictional training fixtures—not production telemetry, a live-model benchmark, a load test, an SLA, or deployment approval.

## Learning objectives and prerequisites

You will validate an NFR contract, inspect p50/p95/p99 and semantic good events, compare cost denominators, model retry amplification, verify safe degradation, enforce agent budgets, and apply owner-controlled gates. Courses 06 and 07 provide the behavioral contract and evidence model.

The execution boundary remains: model and tools may propose; trusted application code validates identity, policy, budgets, targets, mutation, and terminal state.

## Architecture walkthrough

```text
request → auth → context/retrieval → model/tools → validation → storage
   │        │             │              │            │
   └────────┴─────────────┴──────────────┴────────────┘
                  correlated telemetry
                         ↓
 population + measurement + owner target + response policy
```

We measure the whole workflow and selected stages. A dependency failure selects an explicit degradation mode; it never gives the model more authority.

In [ ]:
import copy
import importlib.util
import json
import sys
from pathlib import Path

spec = importlib.util.spec_from_file_location('course08_notebook_lab', Path('lab.py'))
assert spec and spec.loader
lab = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = lab
spec.loader.exec_module(lab)

print('Course 08 fixture:', lab.run_demo()['fixture_status'])

## 1. Contract first: no target invention

A useful NFR names its characteristic, population, measurement, unit, window, owner, target state, evidence method, and failure response. Approved values must match an explicit owner decision. Unresolved targets stay null.

In [ ]:
contract_findings = lab.validate_nfr_contract()
workload_findings = lab.validate_workload_profiles()
measurement_findings = lab.validate_measurement_plan()
assert not contract_findings and not workload_findings and not measurement_findings

contract = lab.load_contract()
target_states = {item['id']: item['target']['status'] for item in contract['requirements']}
print(json.dumps(target_states, indent=2))

The cost and AI-quality objectives remain unresolved. Six agent-budget dimensions also lack owner-approved limits. The contract can still require measurement, ownership, and a blocked gate without inventing ceilings. W2 and W3 are hypotheses, not approved capacity claims. All numeric targets are fictional training values, not recommended production SLOs.

## 2. Baseline measurement: tails, semantic outcomes, and unit economics

The latency boundary is accepted request to completed proposal response. Semantic service success is a valid proposal or an approved degradation that preserves work and reduces autonomy. Compliant workflow success additionally requires governed controls to pass. Cost per successful compliant workflow includes the cost of failed work in the numerator.

In [ ]:
runtime = lab.runtime_measurements()
assert runtime['evidence_status'] == 'synthetic_training_fixture_not_production_evidence'
assert runtime['end_to_end_latency_ms']['denominator'] == 12
assert runtime['semantic_service_success_ratio']['numerator'] == 11
assert runtime['semantic_service_success_ratio']['denominator'] == 12
assert runtime['compliant_workflow_success_ratio']['numerator'] == 11
print(json.dumps(runtime, indent=2))

The p95 is 4800 ms on only twelve synthetic observations. The output carries sample size, source, boundary, and an explicit representativeness warning. This checks calculation and contract plumbing; it does not establish a production SLO. Stage means suggest where time accumulates, but means should not replace stage-tail analysis in production.

## 3. AI quality: inspect slices, not only the aggregate

The included outcomes are fixed. They exercise evaluation plumbing and demonstrate how one weak high-risk slice can hide inside a stronger aggregate.

In [ ]:
quality = lab.quality_measurements()
assert quality['claim'] == 'pipeline_mechanics_only_not_model_quality'
assert quality['overall']['numerator'] == 7
assert quality['overall']['denominator'] == 8
assert quality['slices']['unsupported_field']['value'] == 0.5
print(json.dumps(quality, indent=2))

## 4. Failure injection: retry amplification

The baseline retries every failed request five times. The governed path bounds attempts, opens the circuit, preserves work, and routes all requests to manual processing.

In [ ]:
unsafe = lab.simulate_provider_outage(retry_budget=None, circuit_breaker_threshold=None)
governed = lab.simulate_provider_outage(retry_budget=3, circuit_breaker_threshold=8)
assert unsafe['provider_attempts'] == 600
assert governed['provider_attempts'] == 8
assert governed['work_items_preserved'] == 100
assert governed['circuit_state'] == 'open'
print(json.dumps({'unsafe': unsafe, 'governed': governed}, indent=2))

The circuit makes provider attempts per incoming request fall below one during a complete outage because most requests are diverted before calling the failed dependency. This is controlled load shedding, not improved provider reliability.

## 5. Dependency-specific degradation

Failure criticality determines the response. Authorization and policy uncertainty fail closed. Model failure preserves work with less autonomy. Non-critical analytics may buffer while independent controls continue to govern mutation.

In [ ]:
dependencies = ['authorization', 'model_provider', 'policy_service', 'retrieval', 'analytics_export']
degradation = {name: lab.degradation_decision(name) for name in dependencies}
assert not degradation['authorization']['automatic_mutation']
assert not degradation['policy_service']['automatic_mutation']
assert degradation['analytics_export']['automatic_mutation']
assert all(item['recovery_condition'] for item in degradation.values())
print(json.dumps(degradation, indent=2))

## 6. Failure injection: telemetry exposure and agent budget

A complete trace can still be unsafe if it copies broker content. A plausible model result can still be unsafe if it exceeds trusted execution or side-effect budgets.

In [ ]:
events = lab.load_runtime_events()
leaking = copy.deepcopy(events[0])
leaking['raw_broker_content_logged'] = True
telemetry_codes = {finding.code for finding in lab.telemetry_event_findings(leaking)}
assert 'TELEMETRY_RAW_CONTENT_EXPOSED' in telemetry_codes

looping = copy.deepcopy(events[0])
looping['tool_calls'] = 99  # measured, but unresolved and therefore not silently enforced
looping['authoritative_mutations'] = 2
budget_codes = [finding.code for finding in lab.agent_budget_findings(looping)]
assert budget_codes.count('AGENT_BUDGET_EXCEEDED') == 1
unresolved_budgets = [finding.subject_id for finding in lab.validate_agent_budget()]
assert 'tool_calls' in unresolved_budgets and 'input_tokens' in unresolved_budgets
print({'telemetry': sorted(telemetry_codes), 'budget': budget_codes, 'unresolved': unresolved_budgets})

## 7. Owner-controlled gates and the honest release decision

A measured value does not authorize its own threshold. An approved target without matching evidence is not measured. Synthetic passes remain synthetic.

In [ ]:
gates = lab.target_gates(runtime=runtime, quality=quality)
gate_view = [lab.asdict(gate) for gate in gates]
release = lab.release_assessment(gates, lab.validate_agent_budget(), runtime)
assert sum(item['decision'].value == 'pass' for item in gate_view) == 8
assert release['decision'] == 'blocked_for_production'
assert release['production_ready'] is False
print(json.dumps({'gates': gate_view, 'release': release}, indent=2, default=str))

Eight fixture gates pass, two remain blocked by unauthorized targets, capacity is not measured, and six agent budgets lack owner decisions. Production is also blocked because no production evidence exists. These states are more informative than one composite score.

## Production upgrade

Replace static events with authenticated runtime signals; run representative steady/peak/burst/soak tests; validate provider quotas and backpressure; use durable operation IDs and atomic budget/mutation accounting; execute leakage-controlled model/workflow evaluations with calibrated labels; attest deployed identity/egress policy; and connect misses to owned incident, rollback, or release policies.

## Exercises

1. Add a high-latency event and compare mean, p95, and p99.
2. Convert an approved degradation into an error envelope and recompute semantic availability.
3. Add an unresolved recoverability target with its decision owner and required evidence.
4. Add a French attachment population and show which quality, privacy, performance, and capacity evidence becomes stale.
5. Design a safe terminal state for an unknown mutation outcome.
6. Explain why green synthetic latency cannot be laundered into a production capacity claim.
7. Compare three designs across latency, cost, reliability, privacy, and mutation safety; remove infeasible designs before optimizing.
8. Inject retries into a small subset and show their effect on provider load and tail latency.
9. Double retrieved context on every turn and project token, cost, and latency growth before proposing an owner-approved budget.

## Summary

An executable NFR is a governed measurement contract. It names the population, boundary, statistic, unit, window, owner, target state, evidence, and response. Agentic systems add provider quotas, probabilistic quality, tool/model/token budgets, safe degradation, side-effect limits, cost per compliant outcome, and privacy-aware traces. The right result may be a clear blocker—not a fabricated green release.